# B3 — data-driven hybrid (PI+DD) loss: the last unexercised item-4 variant

**Closing a real gap**: Professor Rabczuk's 2026-10-11 email named 4 candidate methods, the 4th being "Sobolev/H1 training for the pure DD model, and maybe a hybrid model." `rel_l2` (2.51%) and `sobolev` (2.62%) already ran on GPU; `train_B3_data_driven.py --loss hybrid` was built and CPU-verified at the same time but was never actually run.

**What hybrid does**: the loss is `w * (physics term) + (1-w) * (data term)`, where the physics term is the SAME `total_potential_energy_B3` the PI baseline minimizes (no FEM labels needed for this part) and the data term is the plain `rel_l2` supervised loss against the real FEM-labeled training samples. The physics term is divided by a running exponential-moving-average of its own scale (`hybrid_phys_norm=running`) so the two terms stay comparable in magnitude without a hand-picked constant, the same kind of measured-not-guessed calibration this project uses everywhere else. Default `hybrid_phys_weight=0.5` (equal split).

**Reuses the EXISTING training dataset** (`b3_dd_training_set`, 200 samples at `seed=50`, already generated and on Drive from the `rel_l2`/`sobolev` run) -- no new FEM solves needed, only a new training run. Evaluated against the SAME held-out set every other B3 checkpoint uses, now also reporting the H1 semi-norm (added 2026-10-26, see `B3_H1_Seminorm_Reevaluation.ipynb`) alongside the usual rel. L2.

**What to watch**: does hybrid beat both `rel_l2` (2.51%) and `sobolev` (2.62%), matching or exceeding PI's own 1.88%? Or does mixing in a physics term without any real advantage (B3's data-driven labels are already plentiful enough that the physics term adds noise rather than help) make it worse than plain `rel_l2`, as `sobolev` itself already showed (a real, measured negative result -- gradient-matching made things slightly worse for B3, unlike some other contexts)? Either outcome is a genuine, reportable finding completing item 4's own menu.

In [ ]:
# =====================================================================
#  CELL -- B3 data-driven hybrid (PI+DD) loss: REAL training run
#  (50,000 steps, matched to the PI baseline's own budget), reusing the
#  existing b3_dd_training_set dataset (no new FEM solves), then
#  re-evaluated with the H1 semi-norm metric included.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

R = '/content/drive/MyDrive/pfem_run'
TRAIN_DATASET = f'{R}/b3_dd_training_set/dataset.h5'
TEST_DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_DIR = f'{R}/b3_data_driven_hybrid'
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(TRAIN_DATASET), f'expected the existing DD training set at {TRAIN_DATASET}'
print(f'Reusing existing training set: {TRAIN_DATASET}')

sys.argv = [
    'train_B3_data_driven.py',
    '--train_dataset', TRAIN_DATASET,
    '--test_dataset', TEST_DATASET,
    '--out_dir', OUT_DIR,
    '--loss', 'hybrid',
    '--opt_steps', '50000',
    '--batch_size', '8',
    '--eval_every', '2000',
    # hybrid_phys_weight=0.5, hybrid_phys_norm=running are this script's
    # own defaults -- not overridden here.
]
print(f'\nTraining hybrid PI+DD loss, {sys.argv}\n')

t0 = time.time()
from omar_pfem.train_B3_data_driven import main as train_main
train_main()
elapsed = time.time() - t0
print(f'\nTraining finished in {elapsed:.1f}s ({elapsed/3600:.2f}h).')

print(f'\n{"=" * 90}')
print('=== Re-evaluating the best checkpoint with the H1 semi-norm metric included ===')
sys.argv = [
    'reevaluate_B3_data_driven.py',
    '--checkpoint', f'{OUT_DIR}/model_best.pt',
    '--test_dataset', TEST_DATASET,
    '--out_json', f'{OUT_DIR}/qois_h1_reeval.json',
]
from omar_pfem.reevaluate_B3_data_driven import main as reeval_main
reeval_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUT_DIR, kind='b3_data_driven_hybrid',
                    args={'loss': 'hybrid', 'opt_steps': 50000, 'batch_size': 8,
                          'train_dataset': TRAIN_DATASET, 'test_dataset': TEST_DATASET},
                    started_at=_started,
                    results={'elapsed_total_s': time.time() - _started},
                    outputs=[f'{OUT_DIR}/model_best.pt', f'{OUT_DIR}/data_driven_B3_hybrid.json',
                             f'{OUT_DIR}/qois_h1_reeval.json'],
                    notes="Last unexercised variant of Professor Rabczuk's 2026-10-11 item 4 "
                          "(\'Sobolev/H1 training for the pure DD model, and maybe a hybrid "
                          "model\'). rel_l2 (2.51%) and sobolev (2.62%) already ran; this "
                          "completes the menu. Reused the existing b3_dd_training_set dataset "
                          "(no new FEM solves). Evaluated with the H1 semi-norm metric "
                          "(added 2026-10-26) included from the start.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back -- especially the final rel. L2 and '
      'H1 semi-norm numbers, to compare against rel_l2 (2.51%) and sobolev (2.62%).')
